# TrafficFlowBench Task 4 — 가중 Prior NNLS

공개 링크 교통량과 해당 split의 약한 경로 Prior로 **비음수 경로별 교통량**을 추정합니다. CPU 런타임에서 실행하며, 필요한 함수는 모두 아래 셀에 들어 있습니다.

1. Google Drive를 연결하고 이미 받은 데이터의 `RELEASE_ROOT`를 확인합니다.
2. validation에서 Prior 자체·기존 baseline·가중 Prior 후보를 동일한 관측 링크 분할로 비교합니다.
3. 결과표와 그래프를 확인한 뒤, 마지막 설정 셀에서 CSV 생성에 사용할 `λ`와 `p`를 지정합니다.
4. validation·private별 CSV와 두 split을 합친 **Task 4 전용 CSV**를 저장합니다.

**공개 데이터에서는 전체 `S_ODME`를 계산할 수 없습니다.** 아래의 검증 오차는 숨긴 관측 링크에 대한 대리 지표이며, Prior 대비 목적지 분포 변화는 정답 기반 `S_attr`가 아닙니다. Private는 CSV 생성에만 사용합니다.

데이터 폴더에는 `corridors/`와 `task4/`가 있어야 합니다. 저장소 복제·별도 Python 파일·PyTorch·GPU는 필요하지 않습니다.


## 1. 패키지와 Drive

Colab의 **CPU 런타임**에서 실행해 주세요. 이미 받은 데이터를 사용하며, 아래 설치 셀은 Python 패키지만 준비합니다. Drive 대신 런타임에 데이터를 올리셨다면 Drive 연결 셀을 건너뛰고 `RELEASE_ROOT`를 바꾸시면 됩니다.


In [ ]:
%pip -q install numpy pandas scipy matplotlib

from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import time

import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from IPython.display import display

print("NumPy:", np.__version__, "| pandas:", pd.__version__, "| SciPy:", scipy.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 데이터 경로와 비교 설정

`RELEASE_ROOT`는 데이터가 직접 들어 있는 폴더 또는 그 안에 `kaggle_public/`이 있는 부모 폴더를 지정합니다. 기본 경로는 기존 노트북과 같습니다.

`PANELS=None`은 전체 패널입니다. 작은 범위부터 확인하려면 `PANELS=["D12_I5_N"]`으로 바꿔 주세요. `p=0`은 기존 L2, `p=1`은 Prior에 비례하는 분산, `p=2`는 비슷한 상대오차를 가정합니다. 먼저 λ만 비교하려면 `PRIOR_POWERS=(0,)`을 사용합니다. Prior 자체와 기존 `λ=0.05, p=0`은 항상 비교에 포함됩니다.


In [ ]:
RELEASE_ROOT = "/content/drive/MyDrive/dataset/trafficflow"
REPORT_DIR = str(Path(RELEASE_ROOT).parent / "trafficflow_task4_reports")
PANELS = None
LAMBDAS = (0.01, 0.05, 0.2)
PRIOR_POWERS = (0, 1, 2)
PRIOR_FLOOR_RATIO = 0.1
REPEATS = 3
SEED = 42

location = Path(RELEASE_ROOT).expanduser()
release = next((p for p in (location, location / "kaggle_public")
                if (p / "corridors").is_dir() and (p / "task4").is_dir()), None)
if release is None:
    raise FileNotFoundError(f"corridors/와 task4/가 있는 데이터 폴더를 찾지 못했습니다: {location}")
report_dir = Path(REPORT_DIR).expanduser()
print("데이터:", release)
print("결과 폴더:", report_dir)


## 3. 데이터·최적화·검증 함수

아래 함수는 CLI와 같은 가중 Prior NNLS를 노트북 안에서 직접 정의합니다. 관측되지 않은 연결 링크를 교통량 0으로 넣지 않으며, Prior도 현재 split의 파일을 사용합니다.

```text
min f>=0: ||A_measured f - c_measured||² + λ sum_i (f_i-b_i)² / v_i
v_i = (b_i + τ)^p / median_k((b_k + τ)^p)
τ = PRIOR_FLOOR_RATIO × median(positive b)
```

Prior가 모두 0이면 양수 중앙값 대신 1을 사용합니다. 분산의 중앙값은 1로 맞춥니다. `p=0`에서는 기존 baseline과 목적함수가 같습니다.


In [ ]:
ODME_KEYS = ["panel", "departure_time", "path_id"]
ODME_COLUMNS = ODME_KEYS + ["origin_zone", "destination_zone", "path_flow"]


def panels(release: Path, selected: list[str] | None) -> list[str]:
    available = sorted(p.name for p in (release / "corridors").iterdir() if p.is_dir())
    if selected:
        unknown = set(selected) - set(available)
        if unknown:
            raise ValueError(f"Unknown panels: {sorted(unknown)}")
        return sorted(set(selected))
    return available


def stable_seed(*parts):
    return int.from_bytes(hashlib.blake2b("|".join(map(str, parts)).encode(), digest_size=8).digest(), "big")


def family_mean(scores):
    families = {}
    for panel, score in scores.items():
        families.setdefault(panel.rsplit("_", 1)[0], []).append(score)
    return float(np.mean([np.mean(values) for values in families.values()])) if families else 0.0


def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_suffix(path.suffix + ".pending")
    pending.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    pending.replace(path)


In [ ]:
def odme_problem(release: Path, name: str, split: str):
    from scipy.sparse import csr_matrix
    network = release / "corridors" / name / "network"
    root = release / "task4" / name / split
    template = pd.read_csv(root / "sample_submission_path_flow.csv", dtype=str)
    if template.empty or template[ODME_COLUMNS[:-1]].isna().any().any():
        raise ValueError(f"Empty or incomplete ODME template: {name} {split}")
    if not template.panel.eq(name).all() or template.departure_time.nunique() != 1:
        raise ValueError(f"Mismatched ODME template period/panel: {name} {split}")
    if template[ODME_KEYS].duplicated().any() or template.path_id.duplicated().any():
        raise ValueError(f"Duplicate ODME template paths: {name} {split}")
    counts = pd.read_csv(root / "synthetic_link_counts.csv", dtype={"link_id": str})
    prior = pd.read_csv(root / "synthetic_weak_prior.csv", dtype={"path_id": str, "departure_time": str})
    incidence = pd.read_csv(network / "path_link_incidence.csv", dtype={"path_id": str, "link_id": str})
    paths = template.path_id.tolist()
    measured = counts.dropna(subset=["count"])
    if measured.link_id.duplicated().any() or prior.path_id.duplicated().any():
        raise ValueError("Duplicate measured links or priors")
    measured = measured[measured.link_id.isin(incidence.link_id)]
    if measured.empty:
        raise ValueError(f"No measured ODME links: {name} {split}")
    row = {link: i for i, link in enumerate(measured.link_id)}
    column = {path: i for i, path in enumerate(paths)}
    included = incidence[incidence.link_id.isin(row) & incidence.path_id.isin(column)]
    matrix = csr_matrix((np.ones(len(included)),
                         (included.link_id.map(row), included.path_id.map(column))),
                        shape=(len(row), len(column)))
    matrix.data[:] = 1.0  # Repeated incidence rows still mean one traversal.
    prior_values = pd.to_numeric(prior.path_flow, errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(prior_values).all() or (prior_values < 0).any():
        raise ValueError(f"Invalid ODME prior: {name} {split}")
    if "departure_time" in prior and not prior.departure_time.eq(template.departure_time.iloc[0]).all():
        raise ValueError(f"Mismatched ODME prior period: {name} {split}")
    base = pd.Series(prior_values, index=prior.path_id).reindex(paths, fill_value=0).to_numpy(dtype=np.float64)
    observed = pd.to_numeric(measured["count"], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(observed).all() or not np.isfinite(base).all() or (observed < 0).any() or (base < 0).any():
        raise ValueError(f"Invalid counts/prior: {name} {split}")
    return template, matrix, observed, base


def odme_prior_variance(base, power=0, floor_ratio=0.1):
    base = np.asarray(base, dtype=np.float64)
    if base.ndim != 1 or not base.size or not np.isfinite(base).all() or (base < 0).any():
        raise ValueError("ODME prior must be a nonempty finite nonnegative vector")
    if power not in (0, 1, 2) or not np.isfinite(floor_ratio) or floor_ratio <= 0:
        raise ValueError("ODME prior power must be 0, 1, or 2, and floor ratio must be positive")
    if power == 0:
        return np.ones_like(base)
    positive = base[base > 0]
    scale = float(np.median(positive)) if positive.size else 1.0
    variance = (base / scale + floor_ratio) ** power
    variance /= np.median(variance)
    if not np.isfinite(variance).all() or (variance <= 0).any():
        raise ValueError("ODME prior variance is outside the numerical range")
    return variance


def solve_odme(matrix, observed, base, lam, prior_power=0, prior_floor_ratio=0.1):
    from scipy.optimize import lsq_linear, minimize
    from scipy.sparse import csr_matrix
    if lam < 0 or not np.isfinite(lam):
        raise ValueError("ODME lambda must be finite and nonnegative")
    variance = odme_prior_variance(base, prior_power, prior_floor_ratio)
    base = np.asarray(base, dtype=np.float64)
    observed = np.asarray(observed, dtype=np.float64)
    design = csr_matrix(matrix, dtype=np.float64)
    if (observed.ndim != 1 or not observed.size or design.shape != (len(observed), len(base))
            or not np.isfinite(observed).all() or (observed < 0).any()
            or not np.isfinite(design.data).all() or (design.data < 0).any()):
        raise ValueError("Invalid ODME matrix/count dimensions or values")
    # Scaling both vectors preserves the minimizer and avoids large-cost cancellation.
    flow_scale = max(1.0, float(base.max()), float(observed.max()))
    base, observed = base / flow_scale, observed / flow_scale
    if lam == 0:
        result = lsq_linear(design, observed, bounds=(0, np.inf), tol=1e-8, max_iter=2000)
        flow = result.x * flow_scale
        if not result.success or not np.isfinite(flow).all():
            raise RuntimeError(f"ODME solver did not converge: {result.message}")
        return flow
    precision = lam / variance
    if not np.isfinite(precision).all() or (precision <= 0).any():
        raise ValueError("ODME prior precision is outside the numerical range")
    prior_residual = observed - design @ base
    # The dual has one variable per measured link, independent of path count.
    def objective(dual):
        projected = np.asarray(design.T @ dual).ravel()
        correction = np.minimum(projected / precision, base)
        # Centering at the prior removes cancellation between the large linear terms.
        cost = (0.5 * (dual @ dual) + prior_residual @ dual
                + np.dot(projected - 0.5 * precision * correction, correction))
        return cost, dual + prior_residual + design @ correction
    result = minimize(objective, np.zeros(len(observed)), jac=True, method="BFGS",
                      options={"gtol": min(1e-9, 1e-5 / flow_scale), "maxiter": 2000})
    flow = np.maximum(base - np.asarray(design.T @ result.x).ravel() / precision, 0)
    residual, difference = design @ flow - observed, flow - base
    primal = 0.5 * (residual @ residual + np.dot(precision * difference, difference))
    # Clipped flows minimize the Lagrangian; this gap avoids subtracting large costs.
    gap = 0.5 * np.sum((result.x - residual) ** 2)
    flow *= flow_scale
    if (not np.isfinite(flow).all() or not np.isfinite(primal + gap)
            or gap > 1e-7 * ((1 / flow_scale) ** 2 + primal)):
        raise RuntimeError(f"ODME solver did not converge: {result.message}")
    return flow


In [ ]:
def odme_structure(template, matrix):
    columns = matrix.tocsc()
    signatures = {}
    for index in range(columns.shape[1]):
        links = tuple(columns.indices[columns.indptr[index]:columns.indptr[index + 1]])
        signatures[links] = signatures.get(links, 0) + 1
    od_sizes = template.groupby(["origin_zone", "destination_zone"]).size()
    return {"measured_links": matrix.shape[0], "paths": matrix.shape[1],
            "unobserved_paths": int(np.sum(np.diff(columns.indptr) == 0)),
            "distinct_measured_path_signatures": len(signatures),
            "ambiguous_path_groups": sum(size > 1 for size in signatures.values()),
            "paths_in_ambiguous_groups": sum(size for size in signatures.values() if size > 1),
            "od_pairs": len(od_sizes), "od_pairs_with_multiple_paths": int((od_sizes > 1).sum()),
            "max_paths_per_od": int(od_sizes.max()),
            "destinations": int(template.destination_zone.nunique())}


def odme_metrics(template, matrix, observed, base, estimate, fit):
    loaded = np.asarray(matrix @ estimate).ravel()
    metrics = {}
    for label, mask in (("fit", fit), ("hidden", ~fit)):
        residual = loaded[mask] - observed[mask]
        normalized_l1 = float(np.sum(np.abs(residual)) / max(float(observed[mask].sum()), 1e-9))
        rmse = float(np.sqrt(np.mean(residual ** 2)))
        metrics.update({f"{label}_link_normalized_l1": normalized_l1,
                        f"{label}_link_s_link": max(0.0, 1.0 - normalized_l1),
                        f"{label}_link_rmse_vph": rmse,
                        f"{label}_link_normalized_rmse": rmse / max(float(observed[mask].mean()), 1)})
    movement = estimate - base
    prior_total, estimate_total = float(base.sum()), float(estimate.sum())
    destinations = template.destination_zone.to_numpy()
    prior_attr = pd.Series(base).groupby(destinations).sum().to_numpy() / max(prior_total, 1e-9)
    estimate_attr = pd.Series(estimate).groupby(destinations).sum().to_numpy() / max(estimate_total, 1e-9)
    metrics.update({"prior_movement_l1": float(np.abs(movement).sum()),
                    "prior_relative_l1": float(np.abs(movement).sum() / max(prior_total, 1e-9)),
                    "prior_relative_l2": float(np.linalg.norm(movement) / max(np.linalg.norm(base), 1)),
                    "destination_distribution_l1_from_prior": float(np.abs(estimate_attr - prior_attr).sum())
                    if prior_total > 0 and estimate_total > 0 else None,
                    "zero_path_fraction": float(np.mean(estimate == 0))})
    return metrics


In [ ]:
def validate_odme(args):
    if args.repeats < 1 or not args.lambdas or not args.prior_powers:
        raise ValueError("ODME validation needs candidates and at least one repeat")
    if any(not np.isfinite(lam) or lam < 0 for lam in args.lambdas):
        raise ValueError("ODME lambdas must be finite and nonnegative")
    configurations = sorted({(float(lam), power) for lam in args.lambdas for power in args.prior_powers}
                            | {(0.05, 0)})
    candidates = [{"method": "prior", "lambda": None, "prior_power": None}]
    candidates += [{"method": "nnls", "lambda": lam, "prior_power": power} for lam, power in configurations]
    rows, structures = [], {}
    started = time.perf_counter()
    for name in panels(args.release_root, args.panel):
        template, matrix, observed, base = odme_problem(args.release_root, name, args.split)
        if len(observed) < 2:
            raise ValueError("ODME link holdout needs at least two measured links")
        odme_prior_variance(base, 0, args.prior_floor_ratio)
        structures[name] = odme_structure(template, matrix)
        panel_started = time.perf_counter()
        for repeat in range(args.repeats):
            print(f"[검증] {name} — {repeat + 1}/{args.repeats}", flush=True)
            # The first mask stays comparable with earlier lambda-only reports.
            seed = stable_seed(args.seed, name, args.split) if repeat == 0 else stable_seed(args.seed, name, args.split, repeat)
            rng = np.random.default_rng(seed)
            hidden = rng.choice(len(observed), max(1, len(observed) // 5), replace=False)
            fit = np.ones(len(observed), bool)
            fit[hidden] = False
            for candidate in candidates:
                estimate = base.copy() if candidate["method"] == "prior" else solve_odme(
                    matrix[fit], observed[fit], base, candidate["lambda"], candidate["prior_power"], args.prior_floor_ratio)
                rows.append({"panel": name, "repeat": repeat, **candidate,
                             "hidden_link_indices": hidden.tolist(),
                             **odme_metrics(template, matrix, observed, base, estimate, fit)})
        print(f"[패널 완료] {name}: {time.perf_counter() - panel_started:.1f}s", flush=True)
    if not rows:
        raise ValueError("No ODME panels to validate")
    summaries = []
    for candidate in candidates:
        matches = [row for row in rows if all(row[key] == value for key, value in candidate.items())]
        summary = dict(candidate)
        for metric in ("hidden_link_normalized_l1", "hidden_link_normalized_rmse", "hidden_link_s_link",
                       "fit_link_normalized_l1", "prior_movement_l1", "prior_relative_l1", "prior_relative_l2",
                       "destination_distribution_l1_from_prior", "zero_path_fraction"):
            usable = [row for row in matches if row[metric] is not None]
            panel_values = {}
            for row in usable:
                panel_values.setdefault(row["panel"], []).append(row[metric])
            summary[metric] = family_mean({name: float(np.mean(values)) for name, values in panel_values.items()}) if usable else None
        repeat_errors = [family_mean({row["panel"]: row["hidden_link_normalized_l1"] for row in matches
                                     if row["repeat"] == repeat}) for repeat in range(args.repeats)]
        summary["hidden_link_normalized_l1_std"] = float(np.std(repeat_errors))
        summary["worst_panel_hidden_link_normalized_l1"] = max(
            float(np.mean([row["hidden_link_normalized_l1"] for row in matches if row["panel"] == name]))
            for name in structures)
        summaries.append(summary)
    best = min(summaries, key=lambda row: row["hidden_link_normalized_l1"])
    report = {"split": args.split, "proxy_only": True, "selection_metric": "hidden_link_normalized_l1",
              "seed": args.seed, "repeats": args.repeats, "prior_floor_ratio": args.prior_floor_ratio,
              "best_proxy_candidate": {key: best[key] for key in ("method", "lambda", "prior_power")},
              "candidates": summaries, "structure": structures, "panels": rows}
    report["elapsed_seconds"] = time.perf_counter() - started
    write_json(args.output, report)
    return report


def predict_odme(release: Path, split: str, output: Path, lam: float = 0.05,
                 prior_power=0, prior_floor_ratio=0.1, selected=None, prior_only=False) -> None:
    parts = []
    for name in panels(release, selected):
        template, matrix, observed, base = odme_problem(release, name, split)
        solution = base.copy() if prior_only else solve_odme(matrix, observed, base, lam, prior_power, prior_floor_ratio)
        template["path_flow"] = solution
        parts.append(template[ODME_COLUMNS])
    if not parts:
        raise ValueError(f"No ODME panels for {split}")
    result = pd.concat(parts, ignore_index=True)
    output.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output, index=False)


## 4. validation에서 후보 비교

매 반복에서 관측 링크의 약 20%를 숨기며, 같은 패널·반복의 모든 후보는 같은 링크를 숨깁니다. 정규화 L1 오차는 낮을수록 좋습니다. 반복 평균을 패널별로 구한 뒤 방향·도로군을 같은 가중치로 집계합니다.

실행 중에는 패널·반복 진행 상황을 표시합니다. 전체 경로 정답은 공개되지 않아 `best_proxy_candidate`는 링크 복원 기준의 후보일 뿐입니다. 이 후보를 CSV 생성 설정에 자동 적용하지 않습니다.


In [ ]:
validation_args = SimpleNamespace(
    release_root=release, panel=PANELS, split="validation",
    lambdas=list(LAMBDAS), prior_powers=list(PRIOR_POWERS),
    prior_floor_ratio=PRIOR_FLOOR_RATIO, repeats=REPEATS, seed=SEED,
    output=report_dir / "validation_report.json",
)
validation_report = validate_odme(validation_args)
candidate_table = pd.DataFrame(validation_report["candidates"])
repeat_table = pd.DataFrame(validation_report["panels"])
structure_table = pd.DataFrame.from_dict(validation_report["structure"], orient="index").rename_axis("panel").reset_index()
candidate_table.to_csv(report_dir / "candidate_summary.csv", index=False)
repeat_table.to_csv(report_dir / "panel_repeats.csv", index=False)
structure_table.to_csv(report_dir / "network_structure.csv", index=False)
write_json(report_dir / "validation_settings.json", {
    "release_root": str(release), "panels": list(validation_report["structure"]),
    "lambdas": list(LAMBDAS), "prior_powers": list(PRIOR_POWERS),
    "prior_floor_ratio": PRIOR_FLOOR_RATIO, "repeats": REPEATS, "seed": SEED,
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
})
print("링크 복원 오차가 가장 작은 후보:", validation_report["best_proxy_candidate"])
print(f"후보 비교 시간: {validation_report['elapsed_seconds']:.1f}s")
display(candidate_table.sort_values("hidden_link_normalized_l1").reset_index(drop=True))


## 5. 결과 그래프와 구조 진단

그래프의 막대는 숨긴 링크의 정규화 L1 오차 평균, 오차 막대는 반복별 표준편차입니다. 신뢰구간이 아닙니다. `prior_relative_l1`은 Prior 이동량, `destination_distribution_l1_from_prior`는 Prior 대비 목적지 분포 변화입니다. 목적지 총량이 0인 분포의 비교는 `null`입니다.

`unobserved_paths`는 관측 링크를 전혀 지나지 않는 경로, `paths_in_ambiguous_groups`는 같은 관측 링크 조합을 통과하는 경로 수입니다. 동일 조합이 없더라도 모든 경로가 식별되는 것은 아닙니다. `od_pairs_with_multiple_paths=0`이면 OD 정규화가 기존 경로 정규화와 중복될 수 있습니다.


In [ ]:
def candidate_label(row):
    if row["method"] == "prior":
        return "Released prior"
    return f"p={int(row['prior_power'])}, lambda={row['lambda']:g}"

plot_table = candidate_table.sort_values("hidden_link_normalized_l1", ascending=False).copy()
colors = ["#9ca3af" if row["method"] == "prior" else
          "#2563eb" if row["prior_power"] == 0 and row["lambda"] == 0.05 else "#0d9488"
          for _, row in plot_table.iterrows()]
fig, ax = plt.subplots(figsize=(9, max(4, 0.4 * len(plot_table))))
ax.barh([candidate_label(row) for _, row in plot_table.iterrows()],
        plot_table["hidden_link_normalized_l1"],
        xerr=plot_table["hidden_link_normalized_l1_std"], color=colors, capsize=3)
ax.set_xlabel("Held-out link normalized L1 (lower is better; error bars: repeat std)")
ax.set_title("Task 4 observed-link proxy")
fig.tight_layout()
fig.savefig(report_dir / "candidate_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
display(structure_table)
print("상세 결과 폴더:", report_dir)


## 6. CSV 생성 설정

기본값은 기존 baseline인 `λ=0.05, p=0`입니다. 위 결과를 검토한 뒤 아래 값을 직접 바꿔 주세요. 이 설정 셀과 다음 셀만 다시 실행하면 전체 관측 링크로 경로 교통량을 다시 풉니다. Prior 자체를 저장하려면 `PREDICTION_PRIOR_ONLY=True`로 바꿉니다.

`RUN_PREDICTION=True`는 validation·private 양쪽을 생성합니다. `PANELS`를 일부만 지정했다면 CSV도 그 패널만 포함합니다. **출력은 Task 4 전용 파일이며 최종 Kaggle 통합 제출 파일은 아닙니다.**

예측 실행마다 별도 폴더에 저장하며, 양쪽 split·합본·설정이 모두 완성된 경우에만 `predictions/latest.json` を更新します。


In [ ]:
RUN_PREDICTION = True
PREDICTION_PRIOR_ONLY = False
PREDICTION_LAMBDA = 0.05
PREDICTION_POWER = 0


In [ ]:
if RUN_PREDICTION:
    from tempfile import mkdtemp

    selected_panels = panels(release, PANELS)
    all_panels = panels(release, None)
    prediction_root = report_dir / "predictions"
    prediction_root.mkdir(parents=True, exist_ok=True)
    pending_dir = Path(mkdtemp(prefix=".pending-", dir=prediction_root))
    output_dir = pending_dir.with_name("run-" + pending_dir.name.removeprefix(".pending-"))
    csv_parts, artifacts = [], []
    started = time.perf_counter()
    try:
        for split in ("validation", "private"):
            output = pending_dir / split / "path_flow.csv"
            print(f"[CSV 생성] {split}: {len(selected_panels)} panels", flush=True)
            predict_odme(release, split, output, PREDICTION_LAMBDA,
                         PREDICTION_POWER, PRIOR_FLOOR_RATIO, PANELS, PREDICTION_PRIOR_ONLY)
            frame = pd.read_csv(output, dtype={key: str for key in ODME_COLUMNS[:-1]})
            if frame[ODME_KEYS].duplicated().any() or not np.isfinite(frame.path_flow).all() or (frame.path_flow < 0).any():
                raise ValueError(f"CSV 키 또는 예측값이 올바르지 않습니다: {output}")
            csv_parts.append(frame)
            artifacts.append({"split": split, "path": str(output_dir / split / "path_flow.csv"),
                              "rows": len(frame), "sha256": hashlib.sha256(output.read_bytes()).hexdigest()})
            print(f"임시 저장: {output} ({len(frame):,} rows)")
        combined = pd.concat(csv_parts, ignore_index=True)
        if combined[ODME_KEYS].duplicated().any():
            raise ValueError("validation과 private의 ODME 키가 중복됩니다.")
        pending_combined = pending_dir / "path_flow_both.csv"
        combined.to_csv(pending_combined, index=False)
        combined_path = output_dir / "path_flow_both.csv"
        write_json(pending_dir / "settings.json", {
            "method": "prior" if PREDICTION_PRIOR_ONLY else "nnls",
            "lambda": None if PREDICTION_PRIOR_ONLY else PREDICTION_LAMBDA,
            "prior_power": None if PREDICTION_PRIOR_ONLY else PREDICTION_POWER,
            "prior_floor_ratio": PRIOR_FLOOR_RATIO, "release_root": str(release),
            "panels": selected_panels, "all_panels_included": selected_panels == all_panels,
            "files": artifacts, "combined_path": str(combined_path), "combined_rows": len(combined),
            "combined_sha256": hashlib.sha256(pending_combined.read_bytes()).hexdigest(),
            "elapsed_seconds": time.perf_counter() - started,
        })
        # Publish complete runs only; an interrupted retry keeps the previous latest pointer.
        pending_dir.rename(output_dir)
        write_json(prediction_root / "latest.json", {
            "run_dir": str(output_dir), "combined_path": str(combined_path),
            "settings_path": str(output_dir / "settings.json"),
        })
    except BaseException:
        print("저장이 중단됐습니다. 완료 결과는 latest.json을 확인해 주세요.")
        print("새 실행 폴더:", output_dir if output_dir.exists() else pending_dir)
        raise
    print("전체 패널 CSV" if selected_panels == all_panels else "일부 패널만 포함한 CSV")
    print("Task 4 양쪽 split 합본:", combined_path, f"({len(combined):,} rows)")
    print("최근 완료 결과:", prediction_root / "latest.json")
    display(combined.head())
else:
    print("CSV가 필요하면 RUN_PREDICTION=True로 바꾸고 이 셀을 실행해 주세요.")


## 저장 결과

`REPORT_DIR`의 후보 요약·패널별 반복 결과·네트워크 구조 표·그래프·검증 보고서는 같은 이름의 파일을 갱신합니다. 예측 CSV는 실행마다 `predictions/run-<id>/`에 새로 저장합니다. 각 실행 폴더에는 split별 `path_flow.csv`, 양쪽 split의 `path_flow_both.csv`, 설정·행 수·파일 해시를 기록한 `settings.json`이 들어 있습니다.

**최근 완료 결과는 `predictions/latest.json`의 `combined_path`로 확인합니다.** 재실행이 중단되면 이전 완료 결과와 포인터는 유지됩니다. `.pending-<id>/`는 미완료 실행 폴더이므로 제출에 사용하지 마세요.

다른 Task 결과와 합칠 때는 완료된 합본을 공식 `merge_submissions.py`의 `--odme`로 전달합니다. `submission_key.csv`의 모든 필수 행이 채워졌는지 병합 결과를 확인해 주세요.
